# Introduction to Integer Programming

**Course:** IN2040 - Optimization of Organizational Processes  
**Objective:** Recognize when decisions require integer variables, compare integer models with LP relaxations, and interpret solver output.

Follow the lecture sequence: the Dorian advertising model, why rounding can fail, assignment decisions and combinatorial growth, and the LP relaxation. Each example separates the model from its interpretation.

## 1. Set up the modeling tools

PuLP provides a common modeling interface for linear and integer programs. In the first two examples, we will compare models by changing only the variable category.

In [ ]:
%pip install "pulp[cbc]"
import pulp as pl

print(f"PuLP version: {pl.__version__}")

## 2. Dorian advertising: continuous versus whole ads

Dorian buys comedy ads (`comedy_ads`) and football ads (`football_ads`). Each comedy ad costs `$50,000` and each football ad costs `$100,000`. The campaign needs at least 28 units of audience exposure in one segment and 24 in another:

$$7x_1 + 2x_2 \geq 28, \qquad 2x_1 + 12x_2 \geq 24.$$

First solve the LP relaxation, which permits fractional ads. Then require whole ads and compare the objective values.

In [ ]:
def solve_dorian(integer_ads):
    category = "Integer" if integer_ads else "Continuous"
    model = pl.LpProblem("Dorian_Advertising", pl.LpMinimize)
    comedy_ads = model.add_variable("comedy_ads", lowBound=0, cat=category)
    football_ads = model.add_variable("football_ads", lowBound=0, cat=category)
    model += 50_000 * comedy_ads + 100_000 * football_ads
    model += 7 * comedy_ads + 2 * football_ads >= 28
    model += 2 * comedy_ads + 12 * football_ads >= 24

    stats = model.solve(pl.COIN_CMD(msg=False))
    if stats.status != pl.LpSolveStatus.Optimal or not stats.has_solution:
        raise RuntimeError(f"Dorian model did not solve to optimality: {stats.status}")
    return {
        "comedy_ads": pl.value(comedy_ads),
        "football_ads": pl.value(football_ads),
        "cost": pl.value(model.objective),
    }

dorian_lp = solve_dorian(integer_ads=False)
dorian_ip = solve_dorian(integer_ads=True)
print("LP relaxation:", dorian_lp)
print("Integer model:", dorian_ip)
print(f"Integer premium over LP: ${dorian_ip['cost'] - dorian_lp['cost']:,.0f}")

**Interpretation.** The LP relaxation costs `$320,000` at 3.6 comedy ads and 1.4 football ads. Whole advertisements are required in practice, so the integer model costs `$400,000`; `(6, 1)` and `(4, 2)` are both optimal. The relaxation is a lower bound for this minimization problem, not an implementable plan.

## 3. Why simple rounding can fail

Consider an investment problem: maximize the value of projects A and B, subject to a budget of 13 units. Project A uses 7 budget units and earns 21 value units; project B uses 4 and earns 11. Solve the continuous relaxation and the integer model, then compare their solutions.

In [ ]:
def solve_rounding_trap(integer_projects):
    category = "Integer" if integer_projects else "Continuous"
    model = pl.LpProblem("Rounding_Trap", pl.LpMaximize)
    project_a = model.add_variable("project_a", lowBound=0, cat=category)
    project_b = model.add_variable("project_b", lowBound=0, cat=category)
    model += 21 * project_a + 11 * project_b
    model += 7 * project_a + 4 * project_b <= 13

    stats = model.solve(pl.COIN_CMD(msg=False))
    if stats.status != pl.LpSolveStatus.Optimal or not stats.has_solution:
        raise RuntimeError(f"Rounding-trap model did not solve to optimality: {stats.status}")
    return pl.value(project_a), pl.value(project_b), pl.value(model.objective)

portfolio_lp = solve_rounding_trap(integer_projects=False)
portfolio_ip = solve_rounding_trap(integer_projects=True)
print("LP relaxation (A, B, value):", portfolio_lp)
print("Integer optimum (A, B, value):", portfolio_ip)
print("Rounding the LP up to (2, 0) uses 14 budget units, so it is infeasible.")
print("Rounding down to (1, 0) gives value 21, below the integer optimum 33 at (0, 3).")

**Interpretation.** The LP bound is 39 at `(13/7, 0)`, while the integer optimum is 33 at `(0, 3)`. Rounding up violates the budget, and rounding down loses value. An integer model searches feasible discrete decisions instead of rounding a continuous answer.

## 4. Assignment decisions and combinatorial growth

Assign four machines to four tasks, with every machine and every task used exactly once. Let `assign[i, j]` be 1 when machine `i` is assigned to task `j`. There are `4! = 24` one-to-one assignments; the binary model lets a solver find the least-cost assignment without manually listing them.

In [ ]:
machines = ["M1", "M2", "M3", "M4"]
tasks = ["T1", "T2", "T3", "T4"]
setup_time = {
    "M1": {"T1": 13, "T2": 4, "T3": 7, "T4": 6},
    "M2": {"T1": 1, "T2": 11, "T3": 5, "T4": 4},
    "M3": {"T1": 6, "T2": 7, "T3": 2, "T4": 8},
    "M4": {"T1": 1, "T2": 3, "T3": 5, "T4": 9},
}

assignment = pl.LpProblem("Machine_Task_Assignment", pl.LpMinimize)
assign = assignment.add_variable_dicts("assign", (machines, tasks), cat="Binary")
assignment += pl.lpSum(setup_time[i][j] * assign[i][j] for i in machines for j in tasks)
for i in machines:
    assignment += pl.lpSum(assign[i][j] for j in tasks) == 1
for j in tasks:
    assignment += pl.lpSum(assign[i][j] for i in machines) == 1

stats = assignment.solve(pl.COIN_CMD(msg=False))
if stats.status != pl.LpSolveStatus.Optimal or not stats.has_solution:
    raise RuntimeError(f"Assignment model did not solve to optimality: {stats.status}")
chosen = [(i, j) for i in machines for j in tasks if pl.value(assign[i][j]) > 0.5]
print("Optimal assignments:", chosen)
print("Total setup time:", pl.value(assignment.objective), "hours")
print("Number of possible one-to-one assignments:", 4 * 3 * 2 * 1)

**Interpretation.** Each selected binary variable represents a complete assignment decision. The factorial count grows rapidly as the problem expands; optimization solvers use bounds and search strategies to avoid checking every combination explicitly.

## 5. LP relaxation as a bound—and when it is exact

For a maximization problem, relaxing integer restrictions gives an upper bound on the integer optimum. Sometimes the relaxation itself has an integer optimum; in that case, it is also optimal for the IP. Compare the integer model and its continuous relaxation below.

In [ ]:
def solve_relaxation_example(integer_variables):
    category = "Integer" if integer_variables else "Continuous"
    model = pl.LpProblem("LP_Relaxation_Example", pl.LpMaximize)
    x1 = model.add_variable("x1", lowBound=0, cat=category)
    x2 = model.add_variable("x2", lowBound=0, cat=category)
    model += 3 * x1 + 2 * x2
    model += x1 + 2 * x2 <= 5
    model += 2 * x1 + x2 <= 4

    stats = model.solve(pl.COIN_CMD(msg=False))
    if stats.status != pl.LpSolveStatus.Optimal or not stats.has_solution:
        raise RuntimeError(f"Relaxation example did not solve to optimality: {stats.status}")
    return pl.value(x1), pl.value(x2), pl.value(model.objective)

relaxed_solution = solve_relaxation_example(integer_variables=False)
integer_solution = solve_relaxation_example(integer_variables=True)
print("LP relaxation (x1, x2, value):", relaxed_solution)
print("Integer model  (x1, x2, value):", integer_solution)

**Interpretation.** Both models return `(1, 2)` with objective 7. The relaxation is integer in this instance, so its upper bound is attained by a feasible IP solution and proves that solution is optimal. This does not happen for every integer program.

## 6. Summary: choosing and interpreting a model

- Use **continuous** variables when fractional quantities are meaningful.
- Use **integer** variables for counts and **binary** variables for yes/no decisions.
- Use a **mixed-integer** model when continuous quantities depend on discrete choices.
- Do not assume rounding an LP solution will preserve feasibility or optimality.
- Use the LP relaxation as a bound; if its solution is integer, it is optimal for the IP.
- Check solver status and interpret the decision variables in the original context.

Next, apply these ideas to typical IP structures such as knapsack, assignment, bin packing, and set covering.